In [ ]:
import numpy as np

class HDP():
    
    def __init__(self):
        # add all the prior here
        # lambda from H(lambda); This would be some counts; Most likeliy uniform
        # alpha = tendency to open new table
        # gamma = tendency to serve new dish
        # component prior  = uniform or break symmetry?

        self.k = 0              # current number of contexts
        self.max_context = 5   # max number of contexts
        pass
    
    
    def initialize_HDP(self, lambda_H=np.ones(10),TAU=10, gamma=2, alpha=2):
        
        # p(z_t|beta) = [1]
        # p(c_t|pi) = [[1]]
        # gamma_0 = [1]
        # gamma_1 = [Gamma]
        # alpha_0 = [[1]]
        # alpha_1 = [[alpha]]
        # beta = [gamma]
        # Pi = [[]]


        # p(beta|gamma_0, gamma_1)
        # p(pi|alpha_0,alpha_1)
        
        # p(phi,alpha_H)
        
        # p(x|phi)
        
        self.K = 0  # current number of contexts
        self.TAU = TAU
        self.gamma = gamma
        self.alpha = alpha
        self.lambda_H = lambda_H                                                 # parameters of base Measure H = Dir(lambda)
        self.generative_model_counts = np.zeros([lambda_H.size,self.max_context])
        self.generative_model_counts[:,0] = lambda_H                             # int_phi Cat(x|phi)Dir(phi|lambda_H) = Cat(x|lambda_H)  
        self.generative_model_obs = np.nan_to_num(self.generative_model_counts/self.generative_model_counts.sum(axis=0))
        # print(self.generative_model_counts)
        # print(self.generative_model_obs)
        # self.global_prior_context = np.array([1])                              # initialization of beliefs over q(z1)
        self.observations  = np.zeros([self.TAU],dtype=int)


    def initialize_beliefs(self):

        self.prior_context = np.zeros(self.max_context)
        self.prior_context[0] = 1                                                # context prior p(c1)

        self.beta_prime = np.zeros(self.max_context)
        self.global_prior_counts = np.zeros([self.max_context,2])
        self.global_prior = np.zeros(self.max_context)
        self.global_prior[0] = 1

    
        self.transition_matrix_counts = np.zeros([self.max_context, self.max_context,2])
        # self.transition_matrix_counts[0,0] = self.alpha
        # self.transition_matrix = np.divide(self.transition_matrix_counts, self.transition_matrix_counts.sum(axis=0),\
        #                                    out=np.zeros_like(self.transition_matrix_counts),\
        #                                    where=self.transition_matrix_counts.sum(axis=0) != 0)
        self.transition_matrix = np.zeros([self.max_context, self.max_context])
        self.transition_matrix[0,0] = 1 


    def update_beliefs_context(self,obs,tau):
        
        self.observations[tau] = obs

        if tau == 0:
            obs_messages = np.array([self.generative_model_obs[self.observations[tau]], np.ones(self.max_context)])
            q_z = np.array([self.global_prior, np.ones(self.max_context)])

        else:
            obs_messages = self.generative_model_obs[self.observations[tau-1:tau+1]]
            q_z = np.array([self.global_prior, self.global_prior])

        obs_messages = obs_messages*q_z

        obs_message_0 = obs_messages[0,:][:,None]
        obs_message_1 = obs_messages[1,:][None,:]

        q_c1_c2 = self.transition_matrix*self.prior_context[None,:]*obs_message_0*obs_message_1
        print("\nupdating beliefs state")
        print(f"\nPi:\n {self.transition_matrix}")
        # print(f"\nobs_message:\n{obs_messages}")
        # print(f"\nq_z:\n {q_z}")
        # print(f"\np_c1c2:\n {q_c1_c2}")

        posterior_context = (q_c1_c2/q_c1_c2.sum()).sum(axis=1)
        assert np.isclose(posterior_context.sum(),1)

        return posterior_context

        # print("----------------")
        # print(tau, obs,"\n")
        # print(obs_messages)
        # print(q_c1_c2)
        # print(posterior_context,current_context)

    def update_beliefs(self, observation,tau):

        print(f"---------\ntau={tau}")

        if tau == 0:
            self.initialize_beliefs()
        
        posterior_context = self.update_beliefs_context(observation,tau)

        current_context = np.argmax(posterior_context)

        if current_context + 1 > self.K:    # count from 1, not zero
            print("\n*************\nopening new context")
            print("\nadding new k to phi")
            # extend lambda' of q(phi|lambda)
            self.generative_model_counts[:,self.K+1] = self.lambda_H

            print("initializing new beta_k")
            # construct q(z_t = k) = beta'_k * prod_l=1^k-1 (1-beta'_l), where beta'_k = gamma_k1/(gamma_k2)
            self.global_prior_counts[self.K] = [1,self.gamma]                                                       # sample new beta_k
            
            self.beta_prime = np.nan_to_num(self.global_prior_counts/self.global_prior_counts.sum(axis=1)[:,None])  # expected b_k'
            
            beta_prime_l = np.insert(np.cumprod(self.beta_prime,axis=0)[:,1],0,1)                                   #  prod_l=1^k-1 (1-beta'_l)
            beta_prime_k = np.insert(self.beta_prime[:,0], self.K+1, 1)                                             #  beta'_k 

            for k in range(self.K+2):
                self.global_prior[k] = beta_prime_k[k]*beta_prime_l[k]

            print(f"\nexpectation_beta_prime:\n {self.beta_prime}")
            print(f"\nupdating global prior q(c_t|Gamma)")
            print(f"\nq(z_t):\n{self.global_prior}")


            
            print("\ninitializing 2*K+1 new pi_jk'")
            #initialize 2K+1 pi_jk with prior probability (1,alpha)
            self.transition_matrix_counts[self.K,self.K,:] = [1,self.alpha]
            self.pi_prime = np.nan_to_num(self.transition_matrix_counts / self.transition_matrix_counts.sum(axis=-1)[:,:,None]) # expected pi_jk'
            print(f"\nexpectataion pi_jk1'\n {self.pi_prime[:,:,0]}")
            print(f"\nexpectation pi_jk2'\n{self.pi_prime[:,:,1]}")

            print(f"\nupdating global prior q(c_1|c_2, Alpha)")
            # construct q(c_t|c_t-1,alpha) = pi'_jk * prod_l=1^k-1 (1-pi'_jl), where pi'_jk = alpha_jk1/(alpha_jk2)
            pi_prime_l = np.insert(np.cumprod(self.pi_prime, axis=1)[:,:,1], 0, 1, axis=0)
            pi_prime_k = np.insert(self.pi_prime[:,:,0], self.K+1, 1, axis=0)

            for k in range(self.K+2):
                self.transition_matrix[k] = pi_prime_k[k,:]*pi_prime_l[k,:]

            # print(pi_prime_l)
            # print(pi_prime_k)
            # print(self.transition_matrix.sum(axis=0))
            assert np.all(np.isclose(self.transition_matrix.sum(axis=0)[:self.K+1],1))
            
            


            self.K += 1
        else:
            print("did not open a new context")
        
        self.generative_model_counts[observation, current_context] += 1         
        self.generative_model_obs = np.nan_to_num(self.generative_model_counts/self.generative_model_counts.sum(axis=0))

        print("\nupdating lambda of p(x|lambda)")
        print(f"context: {current_context}, obs: {observation}")
        print(f"generative model counts\n {self.generative_model_counts}")
        print(f"generative model data\n {self.generative_model_obs}")


    
        

agent = HDP()
agent.initialize_HDP()

for tau, obs in enumerate([1,2]):
    agent.update_beliefs(obs,tau)
# for obs in [[0,1,3,2], [2,1,3,1]]:
    # agent.update_beliefs(obs)

---------
tau=0

updating beliefs state

Pi:
 [[1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

*************
opening new context

adding new k to phi
initializing new beta_k

expectation_beta_prime:
 [[0.33333333 0.66666667]
 [0.         0.        ]
 [0.         0.        ]
 [0.         0.        ]
 [0.         0.        ]]

updating global prior q(c_t|Gamma)

q(z_t):
[0.33333333 0.66666667 0.         0.         0.        ]

initializing 2*K+1 new pi_jk'

expectataion pi_jk1'
 [[0.33333333 0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]]

expectation pi_jk2'
[[0.66666667 0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0

<ipython-input-6-5402cc662072>:44: RuntimeWarning: invalid value encountered in divide
  self.generative_model_obs = np.nan_to_num(self.generative_model_counts/self.generative_model_counts.sum(axis=0))
<ipython-input-6-5402cc662072>:127: RuntimeWarning: invalid value encountered in divide
  self.beta_prime = np.nan_to_num(self.global_prior_counts/self.global_prior_counts.sum(axis=1)[:,None])  # expected b_k'
<ipython-input-6-5402cc662072>:144: RuntimeWarning: invalid value encountered in divide
  self.pi_prime = np.nan_to_num(self.transition_matrix_counts / self.transition_matrix_counts.sum(axis=-1)[:,:,None]) # expected pi_jk'
<ipython-input-6-5402cc662072>:169: RuntimeWarning: invalid value encountered in divide
  self.generative_model_obs = np.nan_to_num(self.generative_model_counts/self.generative_model_counts.sum(axis=0))


AssertionError: 

In [ ]:
np.arange(4).reshape(2,2)

array([[0, 1],
       [2, 3]])

In [ ]:
a = np.arange(9).reshape(2,2)

ValueError: cannot reshape array of size 9 into shape (2,2)